### Smart Cities & IoT – Real-Time Urban Intelligence Using Databricks
Streaming Sensor Analytics + Grid-Based Spatial Indexing + Delta Live Tables

Smart cities are built on data — massive volumes of live data from air quality sensors, traffic cameras, energy meters, mobility devices, garbage trucks, streetlights, and weather stations. But the “smart” starts happening only when that diverse data is contextualized spatially and temporally to produce meaningful insights for:

Traffic engineering and congestion reduction

Pollution monitoring and alerting

Energy grid optimization

Public safety and emergency response

Urban planning and mobility access

In this article, we’ll walk through how to build an urban intelligence pipeline on Databricks using Bronze/Silver/Gold tables + Spark Structured Streaming + Grid Indexing (no H3).
You’ll see how to ingest IoT data in real time, enrich it with location context, detect anomalies in time windows, and feed downstream dashboards/agents for smart city management.

architecture


### Use Case: Real-Time Air Quality Hotspot Detection
### Objective:

Using streaming air sensor data, identify pollution hotspots in 5-minute windows, enriched with grid IDs and speed (mobility) metrics — and visualize where the air quality is degrading.

We’ll build a pipeline where IoT sensors are sending:

In [0]:
{"sensor_id": "AQ_009", "lat": 37.782, "lon": -122.401, "pm25": 46.3, "ts": 1735914484}


Then we convert raw rows into structured, geo-enabled Silver data and aggregate them to create:

City heatmaps by pm2.5

Anomaly flags per grid and minute

ML features like pollution spikes correlated with vehicle speed around that area

### 1️⃣ BRONZE – Raw Streaming IoT Data
💻 Simulate a Stream of Air Quality Sensor Rows

In [0]:
%sql
-- Create catalog and schemas for Smart City IoT data
CREATE CATALOG IF NOT EXISTS geo_demo;
USE CATALOG geo_demo;

CREATE SCHEMA IF NOT EXISTS smartcity_bronze;
CREATE SCHEMA IF NOT EXISTS smartcity_silver;
CREATE SCHEMA IF NOT EXISTS smartcity_gold;

-- Set default context to bronze for loading
USE SCHEMA smartcity_bronze;


In [0]:
from pyspark.sql.functions import expr, current_timestamp, lit
import random, time

# Generate simple Python dicts
def generate_iot_events(n=100):
    base_lat, base_lon = 37.7749, -122.4194
    now = int(time.time())
    return [
        {
            "sensor_id": f"AQ_{i % 20}",
            "lat": base_lat + random.random() * 0.03,
            "lon": base_lon + random.random() * 0.03,
            "pm25": round(random.uniform(5, 90), 2),
            "ts": now - random.randint(0, 300)   # timestamp in seconds
        }
        for i in range(n)
    ]

# Create DataFrame with schema inference
raw_data = generate_iot_events()
df_raw = spark.createDataFrame(raw_data)

# Add timestamp and metadata
df_bronze = df_raw \
    .withColumn("event_time", expr("timestamp_millis(ts * 1000)")) \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("src_file", lit("simulated_batch"))

# Save as Bronze Delta table
df_bronze.write.format("delta").mode("overwrite").saveAsTable("air_quality")

df_bronze.show(5, truncate=False)


Great — now that the Bronze layer is loaded and saved in smartcity_bronze.air_quality, let’s proceed to the Silver layer, where we:

- ✅ Validate raw data
- ✅ Add GEOGRAPHY points using ST_POINT(lon, lat)
- ✅ Add grid index (grid_id) for fast spatial joins
- ✅ Add partitioning fields like day for dashboards
- ✅ Store the transformed data into smartcity_silver.air_quality

In [0]:
%sql
-- Switch to Silver schema
USE CATALOG geo_demo;
USE SCHEMA smartcity_silver;

-- Create Silver spatial enriched table
CREATE OR REPLACE TABLE air_quality AS
SELECT
  sensor_id,
  lat, lon,
  pm25,
  event_time,
  ST_POINT(lon, lat) AS geo_point,           -- GEOGRAPHY point
  CONCAT(FLOOR(lat * 100), '_', FLOOR(lon * 100)) AS grid_id,    -- Grid tiling index
  CAST(event_time AS DATE) AS day,           -- For partitioning / dashboards
  ingest_ts,
  src_file
FROM geo_demo.smartcity_bronze.air_quality
WHERE lat BETWEEN -90 AND 90
  AND lon BETWEEN -180 AND 180;


Gold Layer – 5-Min Window Aggregation + Hotspot Detection

Now we’ll produce actionable data for analytics. Let’s build:

- A 5-minute window grid aggregation table
- A pollution hotspot table with category values (e.g., MEDIUM, HIGH, SEVERE)

In [0]:
%sql
-- Switch to Gold schema
USE CATALOG geo_demo;
USE SCHEMA smartcity_gold;

-- 5-minute aggregation by grid tile
CREATE OR REPLACE TABLE air_quality_grid_5m AS
SELECT
  grid_id,
  WINDOW(event_time, '5 minutes') AS t,
  AVG(pm25) AS avg_pm25,
  MAX(pm25) AS max_pm25,
  COUNT(*) AS num_events
FROM geo_demo.smartcity_silver.air_quality
GROUP BY grid_id, WINDOW(event_time, '5 minutes');


Let's mark grids with high pollution levels (PM2.5)

In [0]:
%sql
-- Pollution severity table
CREATE OR REPLACE TABLE pollution_hotspots AS
SELECT
  grid_id,
  t.start AS start_time,
  t.end AS end_time,
  avg_pm25,
  CASE 
    WHEN avg_pm25 >= 75 THEN 'SEVERE'
    WHEN avg_pm25 >= 50 THEN 'HIGH'
    WHEN avg_pm25 >= 25 THEN 'MODERATE'
    ELSE 'LOW'
  END AS category
FROM air_quality_grid_5m
WHERE avg_pm25 >= 25  -- Show only at-risk zones
ORDER BY start_time DESC;


You're now ready to either:

- ✅ Build a Dasbhoard in Databricks SQL
- ✅ Export the hotspot table for map rendering (Kepler/PowerBI/Tableau)
- ✅ Feed this into ML models or automation agents (e.g. notify city officials)